In [ ]:
# Cell 1

import os
import sys


from ngsolve import *
import netgen.gui
import time
from netgen.occ import *
from math import sin, cos, pi
import netgen.meshing as ngm
import numpy as np
from netgen.csg import *
from netgen.meshing import MeshingStep


from ngsolve.webgui import Draw
from ngsolve import Mesh
from netgen.occ import SplineApproximation, Pnt, Axis, Face, Wire, Segment, Revolve, OCCGeometry, Z, X, Y
import sympy as sp




# 输入参数都在这里
Ax, Ay, Az = 1, 1, 1  # 椭球在 x/y/z 方向的半轴；当前绕 z 轴旋转的网格构造要求 Ax=Ay
if (Ax, Ay, Az) != (1, 1, 1):
    raise ValueError("Example 6.2 的解析几何要求 Ax=Ay=Az=1")
tau = 5e-2; T = 1  # 固定时间步长、终止时刻；当前 BDF3 要求 T/tau 为整数且 >=3
h = 0.1; r = 1; k = 1;  # h: GenerateMesh 的 maxh；r: Regge 度量阶数；k: 数值解 H1 阶数
metric_bonus_intorder = 10  # gMh.Set 使用的额外求积阶数：在默认阶数上增加，不改变 Regge 阶数 r
quad_order_asm = 30;num_threads_asm = 4
ds_asm = ds(intrules={
    ET.TRIG: IntegrationRule(
        ET.TRIG,
        order=quad_order_asm,
    )
})



# Mesh
# 生成旋转体 和 网格
def BaseTorus(phi): 
    res = Pnt(0, Ay*np.sin(phi),  Az*np.cos(phi))
    # res = Pnt(0, np.sin(phi),  np.cos(phi))
    return res
def GetRotFace(CurveFunc,T_min=0,T_max=np.pi,n=200):
    pnts = [CurveFunc(phi) for phi in np.linspace(T_min,T_max,n)]
    spline = SplineApproximation(pnts, tol=1e-12)
    f = Face(Wire([spline]))
    return f
f = GetRotFace(BaseTorus,T_min=0,T_max=np.pi)
torus = f.Revolve(Axis((0,0,0), Z), 360)
mesh = Mesh(OCCGeometry(torus).GenerateMesh(maxh=h,
            perfstepsend=ngm.MeshingStep.MESHSURFACE,grading=0.5))
Draw(mesh)


# FE Space
# r: 度量的 Regge 阶数；k: 嵌入和速度的 Lagrange 阶数。
fesG = HCurlCurl(mesh, order=r);  fesC = NumberSpace(mesh)
fesV = VectorH1(mesh,order=k); fes = H1(mesh,order=k);
fesMix = fesV*fesC*fesC*fesC*fesC*fesC*fesC


# Time step
# 当前 BDF3 公式使用固定步长；终点必须落在时间网格上。
if not (np.isfinite(tau) and np.isfinite(T) and tau > 0 and T > 0):
    raise ValueError("tau 和 T 必须是有限正数")
N_steps = int(round(T / tau))
if not np.isclose(N_steps * tau, T, rtol=1e-12, atol=0.0):
    raise ValueError("固定步长 BDF3 要求 T/tau 为整数；请设置 tau = T/N_steps")
if N_steps < 3:
    raise ValueError("BDF3 使用 0、tau、2*tau 的初值，要求 T 至少为 3*tau")
tau = T / N_steps  # 仅统一可接受的浮点舍入差异
told = 0.0

folder_name = f"./RateRe/revolution_paper/h_{h:.2e}_tau_{tau:.2e}_T_{T:.2e}_r_{r}_k_{k}"
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
output_path = folder_name




ModuleNotFoundError: No module named 'geometry'

In [ ]:
# Cell 2

surface_region = mesh.Boundaries('.*')  # 三维网格的全部表面；'.*' 是匹配所有边界名称的正则表达式

# [P2] Analytical sphere closest-point map and Jacobian for Regge quadrature.
rho_geometry = sqrt(x*x + y*y + z*z)
a_geometry = CF((x, y, z)) / rho_geometry
Da_geometry = (Id(3) - OuterProduct(a_geometry, a_geometry)) / rho_geometry
q_geometry = -0.32 + 0.48*(1 - a_geometry[2]**2)**2
dq_geometry = -1.92*a_geometry[2]*(1 - a_geometry[2]**2)
# v(p)=(q(pz)*px,q(pz)*py,0)，Q=D_p v；行对应输出分量。
Q_geometry = CF((
    q_geometry, 0, a_geometry[0]*dq_geometry,
    0, q_geometry, a_geometry[1]*dq_geometry,
    0, 0, 0,
), dims=(3, 3))


In [7]:
# Cell 3

# [P3] 真正的 Lagrange 节点插值；%run 使批量驱动也保存插值代码快照。
%run -i surface_lagrange.py
mesh.UnsetDeformation()  # 支持在可视化后重新初始化
def exact_endpoint_values(node_coords):
    p = node_coords / np.linalg.norm(node_coords, axis=1)[:, None]
    q = -0.32 + 0.48*(1 - p[:, 2]**2)**2
    r1 = p.copy()
    r1[:, :2] *= (1 + q)[:, None]
    return np.column_stack((p, r1))

endpoint_coeffs, initial_interp_info = interpolate_surface_lagrange(fes, exact_endpoint_values, k)
Xh0  = GridFunction(fesV); Xh1 = GridFunction(fesV)
for i in range(3):
    Xh0.components[i].vec.FV().NumPy()[:] = endpoint_coeffs[:, i]
    Xh1.components[i].vec.FV().NumPy()[:] = endpoint_coeffs[:, i+3]
print(f"Lagrange endpoint interpolation: max nodal error = {initial_interp_info['max_nodal_error']:.3e}")
Exa_ptr = GridFunction(fesV)
Exa_ptr.vec.data = Xh1.vec - Xh0.vec  # 仅作插值参考；时间推进右端使用解析几何


# [P4] 固定背景度量 g_Mh = Rh(a*g_M)，来自精确球面几何。
nsurf = specialcf.normal(3); NN = OuterProduct(nsurf, nsurf); Ptau = Id(3) - NN
J0_geometry = Da_geometry * Ptau
Jvelocity_geometry = Q_geometry * J0_geometry
J = J0_geometry; G = J.trans * J
gMh = GridFunction(fesG)
gMh.Set(
    G, definedon=mesh.Boundaries('.*'),
    dual=True,
    bonus_intorder=metric_bonus_intorder,
)
g_inv = Ptau * Inv(gMh + NN) * Ptau
sq_det = sqrt(Det(gMh + NN))


# [P3] r(t) 对 t 线性，直接组合系数得到 Ih(r(0)), Ih(r(tau)), Ih(r(2*tau))。
Pos0 = GridFunction(fesV);Pos1 = GridFunction(fesV); 
Pos_1 = GridFunction(fesV);Pos_2 = GridFunction(fesV); 
Pos_hat = GridFunction(fesV)



Solution= GridFunction(fesMix);
SetVisualization(deformation=True)

told = 0.0
Pos_2.vec.data = Xh0.vec
told = tau
Pos_1.vec.data = (1 - told)*Xh0.vec + told*Xh1.vec
told = 2 * tau
step_n = 2  # 当前历史解对应的整数时间步
Pos0.vec.data = (1 - told)*Xh0.vec + told*Xh1.vec





Lagrange endpoint interpolation: max nodal error = 2.220e-16


In [8]:
# Cell 4：Example 6.2 
#
# [P5] 使用解析 J0_geometry、Jvelocity_geometry；保留缓存、约束和 BDF3。
# 本机 NGSolve 6.2.2405 的验证表明：缓存后的应变积分项不要再整体 Compile。

from time import perf_counter as cell4_clock


if step_n == N_steps:
    print("已经推进到终点；如需重新计算，请先重跑 Cell 3。")
elif not 2 <= step_n < N_steps:
    raise RuntimeError("时间步状态不正确，请先运行 Cell 1–3。")
else:
    SetNumThreads(num_threads_asm)

    Exa_ptg = GridFunction(fesG)
    v, alp1, alp2, alp3, beta1, beta2, beta3 = fesMix.TrialFunction()
    w, alp1_t, alp2_t, alp3_t, beta1_t, beta2_t, beta3_t = fesMix.TestFunction()

    Jhat_cached = CacheCF(grad(Pos_hat).Trace())
    g_inv_cached = CacheCF(g_inv)
    mu_cached = CacheCF(sq_det)

    dv = grad(v).Trace()
    dw = grad(w).Trace()
    strain_v = 0.5 * (Jhat_cached.trans * dv + dv.trans * Jhat_cached)
    strain_w = 0.5 * (Jhat_cached.trans * dw + dw.trans * Jhat_cached)

    lhs = BilinearForm(fesMix)
    lhs += (
        2 * Trace(g_inv_cached * strain_v * g_inv_cached * strain_w)
        * mu_cached * ds_asm
    )

    lhs += (
        InnerProduct(w, Cross((alp1, alp2, alp3), Pos_hat)
                     + (beta1, beta2, beta3)) * sq_det
    ).Compile() * ds_asm
    lhs += (
        InnerProduct(v, Cross((alp1_t, alp2_t, alp3_t), Pos_hat)
                     + (beta1_t, beta2_t, beta3_t)) * sq_det
    ).Compile() * ds_asm

    Jhat_rhs = grad(Pos_hat).Trace()
    strain_w_rhs = 0.5 * (Jhat_rhs.trans * dw + dw.trans * Jhat_rhs)
    rhs = LinearForm(fesMix)
    rhs += (
        Trace(g_inv * Exa_ptg * g_inv * strain_w_rhs) * sq_det
    ).Compile() * ds_asm

    # [P5] r(t,p)=p+t*v(p)，J(t)=J0+t*Jv，故 a*partial_t(g)=F0+t*F1。
    J0_rhs = J0_geometry
    Jvelocity_rhs = Jvelocity_geometry
    F0_ptg = J0_rhs.trans * Jvelocity_rhs + Jvelocity_rhs.trans * J0_rhs
    F1_ptg = 2 * Jvelocity_rhs.trans * Jvelocity_rhs
    ptg_const = GridFunction(fesG)
    ptg_slope = GridFunction(fesG)

    cell4_timings = []

    with TaskManager():
        ptg_const.Set(F0_ptg, definedon=surface_region, dual=True,
                      bonus_intorder=metric_bonus_intorder)
        ptg_slope.Set(F1_ptg, definedon=surface_region, dual=True,
                      bonus_intorder=metric_bonus_intorder)

        # Cell 3 已准备第 0、1、2 步；首次从第 3 步求解。
        for n in range(step_n + 1, N_steps + 1):
            told = T if n == N_steps else n * tau

            # 更新同一个 Exa_ptg 对象；rhs 下次组装会自动读到新值。
            Exa_ptg.vec.data = ptg_const.vec + told * ptg_slope.vec

            # 保留原来的三阶外推。
            Pos_hat.vec.data = BaseVector(
                3 * Pos0.vec.FV().NumPy()
                - 3 * Pos_1.vec.FV().NumPy()
                + Pos_2.vec.FV().NumPy()
            )

            # 复用的是表单/表达式对象；矩阵仍随 Pos_hat 改变，每步重新组装。
            stage_start = cell4_clock()
            lhs.Assemble()
            lhs_seconds = cell4_clock() - stage_start
            stage_start = cell4_clock()
            rhs.Assemble()
            rhs_seconds = cell4_clock() - stage_start
            stage_start = cell4_clock()
            Solution.vec.data = lhs.mat.Inverse(inverse="pardiso") * rhs.vec
            solve_seconds = cell4_clock() - stage_start

            # 保留原来的 BDF3 系数，以及先计算新解、再移动历史解的顺序。
            Pos1.vec.data = BaseVector(
                6 / 11 * (
                    3 * Pos0.vec.FV().NumPy()
                    - 3 / 2 * Pos_1.vec.FV().NumPy()
                    + 1 / 3 * Pos_2.vec.FV().NumPy()
                    + tau * Solution.components[0].vec.FV().NumPy()
                )
            )
            Pos_2.vec[:] = Pos_1.vec[:]
            Pos_1.vec[:] = Pos0.vec[:]
            Pos0.vec[:] = Pos1.vec[:]
            step_n = n

            reference_now = (1 - told) * Xh0 + told * Xh1
            errtest = sqrt(Integrate(
                InnerProduct(Pos1 - reference_now, Pos1 - reference_now),
                mesh, definedon=surface_region,
            ))
            cell4_timings.append({
                "step": n, "time": told,
                "lhs_seconds": lhs_seconds,
                "rhs_seconds": rhs_seconds,
                "solve_seconds": solve_seconds,
            })
            print(
                f"h: {h:.6f} t: {told:.6f}, L2_vs_interpolant: {errtest:.6e}; "
                f"LHS: {lhs_seconds:.3f}s, RHS: {rhs_seconds:.3f}s, "
                f"solve: {solve_seconds:.3f}s"
            )


h: 0.100000 t: 0.150000, L2_vs_interpolant: 1.175617e-04; LHS: 5.517s, RHS: 0.600s, solve: 0.149s
h: 0.100000 t: 0.200000, L2_vs_interpolant: 3.152582e-04; LHS: 6.244s, RHS: 0.606s, solve: 0.048s
h: 0.100000 t: 0.250000, L2_vs_interpolant: 5.483488e-04; LHS: 6.154s, RHS: 0.638s, solve: 0.043s
h: 0.100000 t: 0.300000, L2_vs_interpolant: 7.954204e-04; LHS: 5.622s, RHS: 0.559s, solve: 0.040s
h: 0.100000 t: 0.350000, L2_vs_interpolant: 1.051210e-03; LHS: 5.423s, RHS: 0.561s, solve: 0.047s
h: 0.100000 t: 0.400000, L2_vs_interpolant: 1.316522e-03; LHS: 5.602s, RHS: 0.572s, solve: 0.039s
h: 0.100000 t: 0.450000, L2_vs_interpolant: 1.593154e-03; LHS: 5.363s, RHS: 0.565s, solve: 0.051s
h: 0.100000 t: 0.500000, L2_vs_interpolant: 1.882635e-03; LHS: 5.477s, RHS: 0.584s, solve: 0.053s
h: 0.100000 t: 0.550000, L2_vs_interpolant: 2.187980e-03; LHS: 5.710s, RHS: 0.659s, solve: 0.042s
h: 0.100000 t: 0.600000, L2_vs_interpolant: 2.503873e-03; LHS: 5.514s, RHS: 0.670s, solve: 0.056s
h: 0.100000 t: 0.650

In [9]:
# Cell 5: Example 6.2 的连续图范数、终点度量误差与 t=0 初始度量误差。
# 参考解、Jacobian、背景度量均使用解析式；最后的积分仍为数值求积。
import ngsolve as ng
import numpy as np
mesh.UnsetDeformation()  # [P7] 始终在固定 Mh 上测误差，可在绘图后重复运行

if step_n != N_steps or not np.isclose(told, T, rtol=1e-12, atol=0.0):
    raise RuntimeError("请先将时间推进到 T，再计算终点误差")
if not np.allclose((Ax, Ay, Az), (1, 1, 1), rtol=0.0, atol=1e-14):
    raise ValueError("本 Cell 5 专用于 Example 6.2，要求参考曲面 Ax=Ay=Az=1")

# 1. 从固定离散曲面 Mh 到单位球面的精确最近点投影。
# 解析 CF 可以在任意求积点求值，不依赖 Cell 2 的积分点空间或 Xh0/Xh1。
rho_exact = ng.sqrt(ng.x**2 + ng.y**2 + ng.z**2)
p_exact = ng.CF((ng.x/rho_exact, ng.y/rho_exact, ng.z/rho_exact))
Da_cf = (ng.Id(3) - ng.OuterProduct(p_exact, p_exact)) / rho_exact
nh = ng.specialcf.normal(3)
Nh = ng.OuterProduct(nh, nh)
Ph = ng.Id(3) - Nh
J0_exact = Da_cf * Ph

# 2. r(T,p)=((1+T*q(pz))*px, (1+T*q(pz))*py, pz)。
# Q_exact 是速度 (q*px,q*py,0) 对 p 的 Jacobian，行对应输出分量。
q_exact = -0.32 + 0.48*(1 - p_exact[2]**2)**2
dq_exact = -1.92*p_exact[2]*(1 - p_exact[2]**2)
Q_exact = ng.CF((
    q_exact, 0, p_exact[0]*dq_exact,
    0, q_exact, p_exact[1]*dq_exact,
    0, 0, 0,
), dims=(3, 3))
r_minus_l = ng.CF((
    (1 + T*q_exact)*p_exact[0],
    (1 + T*q_exact)*p_exact[1],
    p_exact[2],
))
Jr_exact = (ng.Id(3) + T*Q_exact) * J0_exact

# 3. 误差及对称应变，均在同一个 Mh 的切空间表示。
e = Pos1 - r_minus_l
Jh = ng.grad(Pos1).Trace()
Je = Jh - Jr_exact
S = 0.5*(Jr_exact.trans*Je + Je.trans*Jr_exact)
# 等于 Jh^T Jh - Jr_exact^T Jr_exact；展开式减少接近时的大数相减。
metric_defect = 2*S + Je.trans*Je

# 4. 固定背景为单位球面 g_M，不是随时间变化的目标度量。
# 加 Nh 仅为在三维矩阵表示中补全法向；两侧 Ph 恢复切向逆度量。
G0_exact = J0_exact.trans * J0_exact
H_exact = Ph * ng.Inv(G0_exact + Nh) * Ph
mu_exact = ng.sqrt(ng.Det(G0_exact + Nh))

# 5. t=0：直接用初值 Xh0 与真解 r(0,p)=p 比较，J0_exact 为其精确 Jacobian。
# 使用与终点相同的 g_M 范数、面积权重及求积阶数。
J_initial = ng.grad(Xh0).Trace()
Je_initial = J_initial - J0_exact
# 等于 J_initial^T J_initial - J0_exact^T J0_exact，展开以减少消减误差。
metric_defect_initial = (J0_exact.trans*Je_initial
                         + Je_initial.trans*J0_exact
                         + Je_initial.trans*Je_initial)

error_densities = ng.CF((
    ng.InnerProduct(e, e),
    ng.Trace(H_exact*S*H_exact*S),
    ng.Trace(H_exact*metric_defect*H_exact*metric_defect),
    ng.Trace(H_exact*metric_defect_initial*H_exact*metric_defect_initial),
)) * mu_exact
quad_order_err = 40
_squared_errors = np.asarray(ng.Integrate(
    error_densities, mesh, definedon=mesh.Boundaries('.*'), order=quad_order_err,
), dtype=float)
L2_sq, strain_sq, metric_sq, initial_metric_sq = map(float, _squared_errors)

err_graph = float(np.sqrt(L2_sq + strain_sq))
err_metric = float(np.sqrt(metric_sq))
err_metric_initial = float(np.sqrt(initial_metric_sq))
ndof = fesV.ndof
ndof_total = ndof + 6  # [P6] 三个旋转、三个平移乘子；CSV 的 ndof 仍为嵌入自由度
print(
    f"h: {h:.6e}, dof: {ndof_total} (embedding: {ndof}, multipliers: 6), "
    f"L2error: {np.sqrt(L2_sq):.12e}, strainerr: {np.sqrt(strain_sq):.12e}, "
    f"grapherr: {err_graph:.12e}, metricerr: {err_metric:.12e}, "
    f"initial metric error: {err_metric_initial:.12e}"
)


h: 1.000000e-01, dof: 4227 (embedding: 4221, multipliers: 6), L2error: 7.961012185012e-03, strainerr: 8.995193736618e-02, grapherr: 9.030353675764e-02, metricerr: 1.795120786855e-01, initial metric error: 1.309188191422e-02


In [10]:
# [P8] 显示位置等于 Pos1，位移相对于原始网格坐标。
mesh.UnsetDeformation()
deform = GridFunction(fesV)
deform.Set(Pos1 - CF((x, y, z)), definedon=surface_region)
mesh.SetDeformation(deform)
Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene